# 🦢 Modelo 6: Pre-trained Swin Transformer (Transfer Learning)

En este notebook utilizamos la técnica de **Transfer Learning** con un **Swin Transformer pre-entrenado** en ImageNet. Esto suele dar mejores resultados que entrenar desde cero, ya que el modelo ya conoce formas, texturas y patrones básicos.

Utilizamos la librería `tfswin` para acceder a los pesos oficiales.

---

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
from tensorflow.keras.metrics import AUC

# Añadir el directorio raíz al path para importar módulos locales
sys.path.append('..')
import oct_dataloader as dataloaders
import modelos.modelo_swin_pretrained as swin_model

print("✅ Librerías e importaciones listas")
print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# Configurar GPUs para crecimiento de memoria dinámico
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU(s) detectada(s): {len(gpus)}")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ No se detectó GPU. Se usará la CPU.")

## 1. Carga de Datos
**IMPORTANTE**: Los modelos pre-entrenados del Swin Transformer esperan imágenes de **3 canales (RGB)** y, preferiblemente, un tamaño de **224x224**. Vamos a ajustar el dataloader para esto.

In [ ]:
# Descargar/Localizar dataset
path = kagglehub.dataset_download("anirudhcv/labeled-optical-coherence-tomography-oct")
data_path = path
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'test' in dirs:
        data_path = root
        break

# Configuración para Transfer Learning
IMG_SIZE = (224, 224) # Tamaño estándar para Swin pre-entrenado
BATCH_SIZE = 16       # Los Transformers consumen mucha memoria GPU; bajamos el batch si es necesario

train_ds, val_ds, test_ds, class_names = dataloaders.create_oct_dataloaders(
    data_path=data_path,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',          # Swin pre-entrenado requiere 3 canales (RGB)
    label_mode='categorical',   # Para la métrica AUC
    train_subset_fraction=0.1,  # Subset pequeño para probar la arquitectura rápido
    optimize=True
)

## 2. Definición del Modelo Pre-entrenado
Cargamos `SwinTransformerTiny` con pesos de `ImageNet`.

In [ ]:
# Hiperparámetros para Transfer Learning
INITIAL_LR = 0.0001
DROPOUT = 0.4

# Crear arquitectura con pesos pre-entrenados
model = swin_model.create_pretrained_swin(
    input_shape=(224, 224, 3), # RGB
    num_classes=4, 
    dropout_rate=DROPOUT
)

# Compilar
metrics = [
    'accuracy', 
    AUC(name='auc', multi_label=True)
]

model = swin_model.compile_swin_model(
    model, 
    learning_rate=INITIAL_LR, 
    metrics=metrics
)

model.summary()

## 3. Entrenamiento (Fase 1: Cabezal de Clasificación)
En esta fase el modelo base está congelado.

In [ ]:
EPOCHS = 15

callbacks = swin_model.get_callbacks(
    patience_stop=5, 
    patience_lr=3, 
    factor_lr=0.2
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 4. Visualización

In [ ]:
def plot_history(history):
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Val')
    plt.title('Accuracy')
    plt.legend()
    
    plt.subplot(1, 3, 2)
    plt.plot(history.history['auc'], label='Train')
    plt.plot(history.history['val_auc'], label='Val')
    plt.title('AUC')
    plt.legend()
    
    plt.subplot(1, 3, 3)
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Val')
    plt.title('Loss')
    plt.legend()
    
    plt.show()

plot_history(history)